# NLU Evaluation — LLM-as-a-Judge

Evaluates the nurse shift extraction system in `src/core/webhook/webhook.service.ts`.

The NLU takes a Thai/English nurse message and calls Gemini to extract:
```json
[{"date": "YYYY-MM-DD", "shift": "morning|afternoon|night|leave"}, ...]
```

**Two evaluation layers:**
1. **Exact-match** — precision / recall / F1 on `(date, shift)` pairs (deterministic)
2. **LLM judge** (Groq / Llama 3.3 70B) — holistic score 0–4 + issue explanation (semantic)

**Setup:** Add `GROQ_API_KEY=gsk_...` to your `.env.development` file before running.

In [5]:
%pip install google-genai groq python-dotenv pandas tabulate -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import json
import os
import time
from pathlib import Path

from google import genai
from google.genai import types
from groq import Groq
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

load_dotenv(Path("../.env.development"))

# Collect all available Gemini keys — mirrors production key rotation
GEMINI_KEYS = [
    k for k in [
        os.environ.get("GEMINI_API_KEY"),
        os.environ.get("GEMINI_API_KEY_2"),
        os.environ.get("GEMINI_API_KEY_3"),
        os.environ.get("GEMINI_API_KEY_4"),
    ]
    if k
]

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

if not GEMINI_KEYS:
    raise EnvironmentError("No GEMINI_API_KEY found in .env.development")
if not GROQ_API_KEY:
    raise EnvironmentError(
        "GROQ_API_KEY not found. Add it to .env.development:\n"
        "  GROQ_API_KEY=gsk_..."
    )

REFERENCE_DATE = "2026-05-09"

print(f"{len(GEMINI_KEYS)} Gemini key(s) loaded")
print(f"Groq key loaded: ...{GROQ_API_KEY[-6:]}")
print(f"Reference date:  {REFERENCE_DATE}")

4 Gemini key(s) loaded
Groq key loaded: ...t03KK4
Reference date:  2026-05-09


## Test Dataset

12 cases covering: single shifts, date ranges, relative dates, multi-shift, mixed Thai/English, polite phrasing, and ambiguous input (should return `[]`).

In [7]:
TEST_CASES = [
    {
        "id": "TC01",
        "category": "Single shift — Thai",
        "message": "ขอเวรเช้าวันที่ 15 พฤษภาคมค่ะ",
        "expected": [{"date": "2026-05-15", "shift": "morning"}],
        "notes": "Baseline: simple morning shift request",
    },
    {
        "id": "TC02",
        "category": "Date range leave — Thai",
        "message": "ขอลาวันที่ 20-22 พฤษภาคมนะคะ",
        "expected": [
            {"date": "2026-05-20", "shift": "leave"},
            {"date": "2026-05-21", "shift": "leave"},
            {"date": "2026-05-22", "shift": "leave"},
        ],
        "notes": "Range expansion: 3-day leave",
    },
    {
        "id": "TC03",
        "category": "Night shift next month — Thai",
        "message": "จะขอเวรดึกวันที่ 10 มิถุนายนครับ",
        "expected": [{"date": "2026-06-10", "shift": "night"}],
        "notes": "Night shift in following month",
    },
    {
        "id": "TC04",
        "category": "Two shifts different days — Thai",
        "message": "ขอเวรเช้าวันที่ 1 มิถุนายน และเวรบ่ายวันที่ 2 มิถุนายนค่ะ",
        "expected": [
            {"date": "2026-06-01", "shift": "morning"},
            {"date": "2026-06-02", "shift": "afternoon"},
        ],
        "notes": "Two separate shift requests in one message",
    },
    {
        "id": "TC05",
        "category": "Single shift — English",
        "message": "I want morning shift on May 20",
        "expected": [{"date": "2026-05-20", "shift": "morning"}],
        "notes": "English language request",
    },
    {
        "id": "TC06",
        "category": "Relative date (tomorrow) — Thai",
        "message": "ขอลาพรุ่งนี้ค่ะ",
        "expected": [{"date": "2026-05-10", "shift": "leave"}],
        "notes": "'Tomorrow' relative to 2026-05-09 → 2026-05-10",
    },
    {
        "id": "TC07",
        "category": "Code-switching Thai/English",
        "message": "ขอ night shift วันที่ 25 พฤษภาคมนะครับ",
        "expected": [{"date": "2026-05-25", "shift": "night"}],
        "notes": "Mixed language in same message",
    },
    {
        "id": "TC08",
        "category": "No shift preference (edge case)",
        "message": "สวัสดีค่ะ อยากถามเรื่องตารางเวรหน่อยนะคะ",
        "expected": [],
        "notes": "Greeting/question — should return empty array",
    },
    {
        "id": "TC09",
        "category": "Date before shift keyword — Thai",
        "message": "วันที่ 18 พฤษภาคม ขอเวรบ่ายนะคะ",
        "expected": [{"date": "2026-05-18", "shift": "afternoon"}],
        "notes": "Date appears before shift type in sentence",
    },
    {
        "id": "TC10",
        "category": "Complex multi-request — Thai",
        "message": "ขอเวรเช้าวันที่ 5 มิถุนายน เวรดึกวันที่ 8 มิถุนายน และลาวันที่ 15-16 มิถุนายนค่ะ",
        "expected": [
            {"date": "2026-06-05", "shift": "morning"},
            {"date": "2026-06-08", "shift": "night"},
            {"date": "2026-06-15", "shift": "leave"},
            {"date": "2026-06-16", "shift": "leave"},
        ],
        "notes": "Three requests with a date range — hardest extraction",
    },
    {
        "id": "TC11",
        "category": "Polite Thai with pleasantries",
        "message": "รบกวนขอลาวันที่ 30 พฤษภาคมด้วยนะคะ ขอบคุณมากเลยค่ะ",
        "expected": [{"date": "2026-05-30", "shift": "leave"}],
        "notes": "Polite framing with filler — tests noise robustness",
    },
    {
        "id": "TC12",
        "category": "Date range leave — English",
        "message": "I need leave from June 1 to June 3 please",
        "expected": [
            {"date": "2026-06-01", "shift": "leave"},
            {"date": "2026-06-02", "shift": "leave"},
            {"date": "2026-06-03", "shift": "leave"},
        ],
        "notes": "English date range",
    },
]

print(f"{len(TEST_CASES)} test cases loaded")

12 test cases loaded


## NLU Runner

Mirrors the production `callGemini()` in `webhook.service.ts` — same model, same prompt template, same JSON parsing.

In [8]:
def call_nlu(
    message: str,
    reference_date: str = REFERENCE_DATE,
    model_name: str = "gemini-2.5-flash",
) -> list:
    """Mirrors callGemini() in webhook.service.ts — new google-genai SDK, key rotation."""
    today = f"{reference_date}T00:00:00.000Z"
    prompt = (
        f'Today is {today}. Extract nurse shift preferences from: "{message}".\n'
        'Return a JSON array of objects: {"date": "YYYY-MM-DD", "shift": "morning|afternoon|night|leave"}.'
    )

    last_err = None
    for key in GEMINI_KEYS:
        try:
            client = genai.Client(api_key=key)
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json"
                ),
            )
            raw = response.text.replace("```json", "").replace("```", "").strip()
            return json.loads(raw)
        except Exception as e:
            last_err = e
            continue

    raise RuntimeError(f"All {len(GEMINI_KEYS)} Gemini keys failed. Last error: {last_err}")


# Smoke test
sample = call_nlu("ขอเวรเช้าวันที่ 15 พฤษภาคมค่ะ")
print("Smoke test output:", json.dumps(sample, ensure_ascii=False))

RuntimeError: All 4 Gemini keys failed. Last error: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}

## Exact-Match Evaluator

Computes precision / recall / F1 on `(date, shift)` pairs. No LLM involved — deterministic.

In [ ]:
def exact_match_eval(expected: list, actual: list) -> dict:
    expected_set = {(item["date"], item["shift"]) for item in expected}
    actual_set   = {(item["date"], item["shift"]) for item in actual}

    tp = expected_set & actual_set
    fp = actual_set - expected_set   # hallucinated
    fn = expected_set - actual_set   # missed

    # Edge case: both empty → perfect score
    if not expected_set and not actual_set:
        precision = recall = f1 = 1.0
    else:
        precision = len(tp) / len(actual_set)   if actual_set   else 0.0
        recall    = len(tp) / len(expected_set) if expected_set else 0.0
        f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return {
        "precision": round(precision, 3),
        "recall":    round(recall, 3),
        "f1":        round(f1, 3),
        "tp": sorted(tp),
        "fp": sorted(fp),  # false positives = hallucinations
        "fn": sorted(fn),  # false negatives = missed extractions
    }

## LLM Judge (Groq / Llama 3.3 70B)

Evaluates extraction quality holistically — catches errors exact-match misses (e.g. off-by-one dates, wrong month inference, plausible-but-wrong shift types).

Uses `response_format={"type": "json_object"}` to guarantee valid JSON output.

In [ ]:
JUDGE_SYSTEM = """You are an expert evaluator for a Thai hospital nurse shift scheduling chatbot.

The chatbot's NLU extracts structured shift preferences from nurse messages (Thai or English).
Valid shifts: morning, afternoon, night, leave
Date format: YYYY-MM-DD

Your job: given an input message, the ground-truth output, and the system's actual output,
decide how well the system understood the nurse's request.

Score guide:
  4 — Perfect: all dates correct, all shifts correct, nothing extra
  3 — Mostly correct: minor error (e.g. off-by-one in range, wrong day name)
  2 — Partial: got some pairs right, missed or misidentified others
  1 — Mostly wrong: significant errors throughout
  0 — Complete failure: empty when should have extracted, or entirely hallucinated

Return ONLY valid JSON with no markdown fences:
{
  "holistic_score": <0-4>,
  "is_usable": <true/false>,
  "date_parsing_ok": <true/false>,
  "shift_classification_ok": <true/false>,
  "hallucination_present": <true/false>,
  "issues": ["specific problem 1", ...],
  "verdict": "<one concise sentence>"
}"""


def llm_judge(
    message: str,
    expected: list,
    actual: list,
    reference_date: str = REFERENCE_DATE,
    client: Groq = None,
) -> dict:
    if client is None:
        client = Groq(api_key=GROQ_API_KEY)

    user_prompt = (
        f"Reference date (today): {reference_date}\n\n"
        f"Nurse message:\n\"{message}\"\n\n"
        f"Ground truth:\n{json.dumps(expected, ensure_ascii=False, indent=2)}\n\n"
        f"System output:\n{json.dumps(actual, ensure_ascii=False, indent=2)}\n\n"
        "Evaluate and return JSON."
    )

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        max_tokens=512,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user",   "content": user_prompt},
        ],
    )
    return json.loads(response.choices[0].message.content)


# Smoke test
test_judgment = llm_judge(
    message="ขอเวรเช้าวันที่ 15 พฤษภาคมค่ะ",
    expected=[{"date": "2026-05-15", "shift": "morning"}],
    actual=sample,
)
print("Judge smoke test:", json.dumps(test_judgment, ensure_ascii=False, indent=2))

## Run Full Evaluation

In [ ]:
judge_client = Groq(api_key=GROQ_API_KEY)
results = []

for tc in TEST_CASES:
    print(f"[{tc['id']}] {tc['category']}")

    # 1. Call the NLU
    nlu_error = None
    try:
        actual = call_nlu(tc["message"])
    except Exception as e:
        actual = []
        nlu_error = str(e)
        print(f"  ⚠ NLU error: {e}")

    # 2. Exact-match metrics
    exact = exact_match_eval(tc["expected"], actual)

    # 3. LLM judge
    judge_error = None
    try:
        judgment = llm_judge(tc["message"], tc["expected"], actual, client=judge_client)
    except Exception as e:
        judgment = {
            "holistic_score": -1,
            "is_usable": False,
            "hallucination_present": False,
            "issues": [],
            "verdict": f"Judge error: {e}",
        }
        judge_error = str(e)
        print(f"  ⚠ Judge error: {e}")

    print(
        f"  F1={exact['f1']:.2f}  "
        f"Judge={judgment['holistic_score']}/4  "
        f"Usable={'✓' if judgment.get('is_usable') else '✗'}  "
        f"{judgment.get('verdict', '')}"
    )

    results.append({
        "id":            tc["id"],
        "category":      tc["category"],
        "message":       tc["message"],
        "expected":      tc["expected"],
        "actual":        actual,
        "notes":         tc["notes"],
        # exact-match
        "precision":     exact["precision"],
        "recall":        exact["recall"],
        "f1":            exact["f1"],
        "fp":            exact["fp"],
        "fn":            exact["fn"],
        # judge
        "judge_score":   judgment["holistic_score"],
        "is_usable":     judgment.get("is_usable", False),
        "hallucination": judgment.get("hallucination_present", False),
        "issues":        judgment.get("issues", []),
        "verdict":       judgment.get("verdict", ""),
        # errors
        "nlu_error":     nlu_error,
        "judge_error":   judge_error,
    })

    time.sleep(0.5)  # avoid Gemini rate limits

print("\n✅ Evaluation complete")

## Results Table

In [ ]:
df = pd.DataFrame(results)

display_df = df[[
    "id", "category",
    "precision", "recall", "f1",
    "judge_score", "is_usable", "hallucination",
    "verdict",
]].copy()

# Colour F1 and judge score green→red
styled = (
    display_df.style
    .background_gradient(subset=["f1"], cmap="RdYlGn", vmin=0, vmax=1)
    .background_gradient(subset=["judge_score"], cmap="RdYlGn", vmin=0, vmax=4)
    .format({"precision": "{:.2f}", "recall": "{:.2f}", "f1": "{:.2f}", "judge_score": "{:.0f}"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
)

display(styled)

## Summary Statistics

In [ ]:
n = len(df)
perfect   = (df["f1"] == 1.0).sum()
usable    = df["is_usable"].sum()
halluc    = df["hallucination"].sum()
mean_f1   = df["f1"].mean()
mean_judge = df[df["judge_score"] >= 0]["judge_score"].mean()

print("=" * 50)
print(f"  Test cases        : {n}")
print(f"  Perfect F1 (=1.0) : {perfect}/{n}")
print(f"  Usable (judge)    : {usable}/{n}")
print(f"  Hallucinations    : {halluc}/{n}")
print(f"  Mean exact F1     : {mean_f1:.3f}")
print(f"  Mean judge score  : {mean_judge:.2f} / 4")
print("=" * 50)

# Breakdown by failure type
missed  = df[df["fn"].apply(len) > 0]
extra   = df[df["fp"].apply(len) > 0]
print(f"\nCases with missed extractions : {len(missed)}")
print(f"Cases with hallucinations     : {len(extra)}")

## Failure Analysis

In [ ]:
failures = df[df["f1"] < 1.0].sort_values("f1")

if failures.empty:
    print("All test cases passed exact-match evaluation.")
else:
    print(f"{len(failures)} failed test case(s):\n")
    for _, row in failures.iterrows():
        print(f"{'─'*60}")
        print(f"[{row['id']}] {row['category']}")
        print(f"  Message  : {row['message']}")
        print(f"  Expected : {json.dumps(row['expected'], ensure_ascii=False)}")
        print(f"  Actual   : {json.dumps(row['actual'],   ensure_ascii=False)}")
        print(f"  F1={row['f1']:.2f}  Judge={row['judge_score']}/4")
        if row["fn"]:
            print(f"  Missed   : {row['fn']}")
        if row["fp"]:
            print(f"  Extra    : {row['fp']}")
        if row["issues"]:
            for issue in row["issues"]:
                print(f"  Issue    : {issue}")
        print(f"  Verdict  : {row['verdict']}")

## Judge vs Exact-Match Agreement

Cases where the two evaluators disagree are the most interesting — they reveal where exact-match is too strict (e.g. semantically correct but different date format) or the judge is too lenient.

In [ ]:
df["exact_pass"]  = df["f1"] == 1.0
df["judge_pass"]  = df["judge_score"] >= 3
df["disagree"]    = df["exact_pass"] != df["judge_pass"]

disagree = df[df["disagree"]]
if disagree.empty:
    print("Exact-match and judge agree on all cases.")
else:
    print(f"{len(disagree)} disagreement(s):\n")
    for _, row in disagree.iterrows():
        direction = "exact=✓ judge=✗" if row["exact_pass"] else "exact=✗ judge=✓"
        print(f"  [{row['id']}] {direction}  |  {row['verdict']}")

# Correlation between the two scores
corr = df[["f1", "judge_score"]].corr().iloc[0, 1]
print(f"\nPearson correlation (F1 ↔ judge score): {corr:.3f}")

## Export Results to JSON

In [ ]:
from datetime import datetime as dt

output = {
    "run_at": dt.now().isoformat(),
    "reference_date": REFERENCE_DATE,
    "summary": {
        "total": n,
        "perfect_f1": int(perfect),
        "usable": int(usable),
        "hallucinations": int(halluc),
        "mean_f1": round(float(mean_f1), 4),
        "mean_judge_score": round(float(mean_judge), 4),
    },
    "results": [
        {
            "id": r["id"],
            "category": r["category"],
            "message": r["message"],
            "expected": r["expected"],
            "actual": r["actual"],
            "f1": r["f1"],
            "judge_score": r["judge_score"],
            "is_usable": r["is_usable"],
            "hallucination": r["hallucination"],
            "issues": r["issues"],
            "verdict": r["verdict"],
        }
        for r in results
    ],
}

out_path = Path("nlu_eval_results.json")
out_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Results saved to {out_path.resolve()}")